# Magnus Carlsen Performance Analysis — Data Cleaning

Analysis of 9,336 of Magnus Carlsen's Chess.com games (2014-2026), fetched via 
the public Chess.com API. This notebook cleans and prepares the raw data — 
handling missing values, simplifying categories, and engineering features 
needed to investigate his opening performance, fatigue, tilt, and resilience 
under pressure in the follow-up analysis notebook.

In [1]:
import pandas as pd
import numpy as np
games=pd.read_csv("magnus_games.csv")

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [3]:
games

,url,time_class,time_control,rated,rules,end_time,eco_url,opening,white_username,white_rating,...,black_username,black_rating,black_result,magnus_color,magnus_rating,opponent_rating,magnus_result,white_accuracy,black_accuracy,magnus_accuracy
0,https://www.chess.com/game/live/998800720,rapid,1500+25,False,chess,1418582233,https://www.chess.com/openings/Modern-Defense-...,Modern Defense Bishop Attack 3...d6 4.Nf3,MagnusCarlsen,2862,...,RainnWilson,1200,checkmated,white,2862,1200,win,98.46,82.04,98.46
1,https://www.chess.com/game/live/998800982,rapid,1500+25,False,chess,1418582613,https://www.chess.com/openings/Nimzo-Indian-De...,Nimzo Indian Defense Hubner Rubinstein Variati...,MagnusCarlsen,2862,...,solskytz,1702,resigned,white,2862,1702,win,96.56,80.40,96.56
2,https://www.chess.com/game/live/998802049,rapid,1500+25,False,chess,1418583369,https://www.chess.com/openings/Ruy-Lopez-Openi...,Ruy Lopez Opening Berlin lHermet Variation 6.B...,MagnusCarlsen,2862,...,Tildenbeatsu,1200,checkmated,white,2862,1200,win,96.52,76.70,96.52
3,https://www.chess.com/game/live/998800852,rapid,1500+25,False,chess,1418583952,https://www.chess.com/openings/French-Defense-...,French Defense Tarrasch Guimard Main Line...6....,MagnusCarlsen,2862,...,mtmnfy,1200,resigned,white,2862,1200,win,94.12,85.41,94.12
4,https://www.chess.com/game/live/998801568,rapid,1500+25,False,chess,1418584335,https://www.chess.com/openings/Trompowsky-Atta...,Trompowsky Attack 2...c5 3.d5 Ne4,MagnusCarlsen,2862,...,stepanosinovsky,2360,win,white,2862,2360,resigned,83.55,87.73,83.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9331,https://www.chess.com/game/live/169139813150,blitz,180,True,chess,1779569185,https://www.chess.com/openings/Nimzo-Indian-De...,Nimzo Indian Defense Kmoch Variation 4...c5 5.a3,MagnusCarlsen,3373,...,hakanazeri2,3061,resigned,white,3373,3061,win,94.23,84.14,94.23
9332,https://www.chess.com/game/live/169410789930,blitz,180,True,chess,1780061104,https://www.chess.com/openings/Pirc-Defense-2....,Pirc Defense 2.d4 Nf6 3.Bd3 e5 4.Nf3,Resentiment18,3034,...,MagnusCarlsen,3375,win,black,3375,3034,win,81.43,89.88,89.88
9333,https://www.chess.com/game/live/169410964060,blitz,180,True,chess,1780061380,https://www.chess.com/openings/English-Opening...,English Opening Kings English Nimzowitsch Floh...,MagnusCarlsen,3377,...,Resentiment18,3032,resigned,white,3377,3032,win,81.37,73.50,81.37
9334,https://www.chess.com/game/live/169411164842,blitz,180,True,chess,1780061522,https://www.chess.com/openings/Philidor-Defens...,Philidor Defense Exchange Variation...6.Bf4 O ...,Resentiment18,3030,...,MagnusCarlsen,3379,win,black,3379,3030,win,80.18,96.35,96.35


In [4]:
games['rules'].value_counts()

rules
chess         9018
chess960       311
oddschess        5
threecheck       2
Name: count, dtype: int64

In [5]:
games.groupby('rules')['time_class'].value_counts().unstack()

time_class,blitz,bullet,rapid
rules,,,
chess,6572.0,2115.0,331.0
chess960,304.0,7.0,NaN
oddschess,1.0,NaN,4.0
threecheck,2.0,NaN,NaN


___

# DATA CLEANING

___

## 1. Checking duplicate rows

In [6]:
games[games.duplicated()]

,url,time_class,time_control,rated,rules,end_time,eco_url,opening,white_username,white_rating,...,black_username,black_rating,black_result,magnus_color,magnus_rating,opponent_rating,magnus_result,white_accuracy,black_accuracy,magnus_accuracy


In [7]:
games.shape

(9336, 21)

___

## 2. Checking for missing values

In [8]:
games.isnull().sum()

url                  0
time_class           0
time_control         0
rated                0
rules                0
end_time             0
eco_url            316
opening            316
white_username       0
white_rating         0
white_result         0
black_username       0
black_rating         0
black_result         0
magnus_color         0
magnus_rating        0
opponent_rating      0
magnus_result        0
white_accuracy      10
black_accuracy      10
magnus_accuracy     10
dtype: int64

### Handling missing opening data
316 games (~3.4%) are missing opening/ECO information from the API. 
Rather than dropping these rows entirely (which would lose valid data 
for other analyses like rating trends and session fatigue), I'm leaving 
them as NaN. Pandas automatically excludes NaN values in groupby operations, 
so opening-specific analysis remains clean while these games still 
contribute to all other questions.

### Accuracy value handling

In [9]:
games[games[['white_accuracy','black_accuracy','magnus_accuracy']].isnull().all(axis=1)]

,url,time_class,time_control,rated,rules,end_time,eco_url,opening,white_username,white_rating,...,black_username,black_rating,black_result,magnus_color,magnus_rating,opponent_rating,magnus_result,white_accuracy,black_accuracy,magnus_accuracy
201,https://www.chess.com/game/live/2415839401,blitz,180+2,True,chess,1510088860,https://www.chess.com/openings/Undefined,Undefined,MagnusCarlsen,3037,...,Storne89,2532,resigned,white,3037,2532,win,NaN,NaN,NaN
903,https://www.chess.com/game/live/61860050863,blitz,180,True,threecheck,1668121356,https://www.chess.com/openings/Pirc-Defense,Pirc Defense,LacusSomniorum,3304,...,MagnusCarlsen,3042,resigned,black,3042,3304,resigned,NaN,NaN,NaN
2275,https://www.chess.com/game/live/71945172533,blitz,180+1,True,chess,1678206297,https://www.chess.com/openings/Undefined,Undefined,Smirnov_Pavel,2762,...,MagnusCarlsen,3223,win,black,3223,2762,win,NaN,NaN,NaN
2346,https://www.chess.com/game/live/73149933575,blitz,180+1,True,chess,1679410984,https://www.chess.com/openings/Undefined,Undefined,deribas12345,2578,...,MagnusCarlsen,3199,win,black,3199,2578,win,NaN,NaN,NaN
2709,https://www.chess.com/game/live/77406827095,blitz,180+1,True,chess,1683668304,https://www.chess.com/openings/Undefined,Undefined,MagnusCarlsen,3269,...,OK97,2819,resigned,white,3269,2819,win,NaN,NaN,NaN
2711,https://www.chess.com/game/live/77408534269,blitz,180+1,True,chess,1683669935,https://www.chess.com/openings/Kings-Indian-De...,Kings Indian Defense Semi Classical Variation ...,MagnusCarlsen,3274,...,Sargsyan_Shant,3002,resigned,white,3274,3002,win,NaN,NaN,NaN
2800,https://www.chess.com/game/live/78864833399,blitz,180,True,threecheck,1685126190,https://www.chess.com/openings/Nimzowitsch-Def...,Nimzowitsch Defense French Connection Variation,MagnusCarlsen,2619,...,NikoTheodorou,2580,win,white,2619,2580,threecheck,NaN,NaN,NaN
2904,https://www.chess.com/game/live/80413358415,blitz,180+1,True,chess,1686674866,https://www.chess.com/openings/French-Defense-...,French Defense Winawer Variation 4.Qd3 dxe4 5....,Oleksandr_Bortnyk,2959,...,MagnusCarlsen,3296,win,black,3296,2959,win,NaN,NaN,NaN
2983,https://www.chess.com/game/live/81642261179,blitz,180+1,True,chess,1687903480,https://www.chess.com/openings/Nimzo-Indian-De...,Nimzo Indian Defense Reshevsky Variation 5...Re8,Sanan_Sjugirov,2953,...,MagnusCarlsen,3300,win,black,3300,2953,win,NaN,NaN,NaN
7902,https://www.chess.com/game/live/152613998555,blitz,180+1,True,chess960,1758300423,NaN,NaN,MagnusCarlsen,2791,...,ant_chess_2019,2337,win,white,2791,2337,timeout,NaN,NaN,NaN





10 games are missing accuracy data entirely for both players. Since accuracy 
is central to several questions in this analysis (clean vs messy games, 
tilt, upsets), these rows are dropped rather than filled — imputing accuracy 
would introduce fabricated values into a metric the whole analysis depends on.




In [10]:
null_white  = games[games['white_accuracy'].isnull()].index
null_black  = games[games['black_accuracy'].isnull()].index
null_magnus = games[games['magnus_accuracy'].isnull()].index

print(null_white)
print(null_black)
print(null_magnus)

Index([201, 903, 2275, 2346, 2709, 2711, 2800, 2904, 2983, 7902], dtype='int64')
Index([201, 903, 2275, 2346, 2709, 2711, 2800, 2904, 2983, 7902], dtype='int64')
Index([201, 903, 2275, 2346, 2709, 2711, 2800, 2904, 2983, 7902], dtype='int64')


In [11]:
games = games.dropna(subset=['white_accuracy','black_accuracy','magnus_accuracy'], how='all').copy()


In [12]:
games[['white_accuracy','black_accuracy','magnus_accuracy']].isnull().sum()


white_accuracy     0
black_accuracy     0
magnus_accuracy    0
dtype: int64

In [13]:
games.shape

(9326, 21)

In [14]:
games.duplicated().sum()

np.int64(0)

---

## 3. Converting timestamps
`end_time` is stored as a Unix timestamp (UTC). Converting to datetime enables 
chronological sorting — required for session detection and the tilt analysis.

In [15]:
games["date"] = pd.to_datetime(games["end_time"], unit="s")
games['date']

0      2014-12-14 18:37:13
1      2014-12-14 18:43:33
2      2014-12-14 18:56:09
3      2014-12-14 19:05:52
4      2014-12-14 19:12:15
               ...        
9331   2026-05-23 20:46:25
9332   2026-05-29 13:25:04
9333   2026-05-29 13:29:40
9334   2026-05-29 13:32:02
9335   2026-05-29 13:37:30
Name: date, Length: 9326, dtype: datetime64[ns]

Extracting `year` to track performance trends over Magnus's career.

In [16]:
games['year']=games['date'].dt.year
games['year']

0       2014
1       2014
2       2014
3       2014
4       2014
        ... 
9331    2026
9332    2026
9333    2026
9334    2026
9335    2026
Name: year, Length: 9326, dtype: int32

____

## 4. Simplifying game outcomes
Chess.com records 11 distinct result types (resigned, timeout, stalemate, etc.). 
Collapsing these into win/loss/draw makes outcome-based analysis tractable.

In [17]:
games['magnus_result'].value_counts()

magnus_result
win                   6542
resigned              1152
repetition             580
timeout                345
insufficient           285
agreed                 147
checkmated             109
timevsinsufficient      72
stalemate               66
abandoned               14
50move                  14
Name: count, dtype: int64

In [18]:
conditions=[
    games['magnus_result']=="win",
    games['magnus_result'].isin(['resigned','timeout','checkmated','abandoned'])
]

choices=["win","loss"]

games['magnus_result_simple']=np.select(conditions, choices, default="draw")

In [19]:
games.loc[:,['magnus_result','magnus_result_simple']]

,magnus_result,magnus_result_simple
0,win,win
1,win,win
2,win,win
3,win,win
4,resigned,loss
...,...,...
9331,win,win
9332,win,win
9333,win,win
9334,win,win


In [20]:
games['magnus_result_simple'].shape

(9326,)

___
## 5. Tilt setup
To test whether a loss affects Magnus's next game, each game needs to know 
the outcome of the *previous* game. Requires the dataset sorted chronologically 
(verified below) before shifting.

In [21]:
games['prev_result']=games['magnus_result_simple'].shift(1)

In [22]:
games.loc[11:23,['magnus_result_simple','prev_result']]

,magnus_result_simple,prev_result
11,win,win
12,win,win
13,draw,win
14,win,draw
15,win,win
16,loss,win
17,win,loss
18,draw,win
19,win,draw
20,loss,win


___
## 6. Simplifying openings
Raw opening names include full move sequences (e.g. "Modern Defense Bishop 
Attack 3...d6 4.Nf3"), fragmenting nearly identical openings into hundreds of 
unique values. Extracting the first two words groups games into their broader 
opening family, enabling meaningful frequency and win-rate analysis.

In [23]:
games['opening'].unique()

array(['Modern Defense Bishop Attack 3...d6 4.Nf3',
       'Nimzo Indian Defense Hubner Rubinstein Variation 5...d5 6.a3 Bxc3 7.Nxc3',
       'Ruy Lopez Opening Berlin lHermet Variation 6.Bg5 Be7 7.Bxe7', ...,
       'English Opening Fianchetto Ultra Symmetrical Line 5.Nf3 d6 6.O O Bd7',
       'Nimzo Indian Defense Kmoch Variation 4...c5 5.a3',
       'English Opening Kings English Nimzowitsch Flohr Variation 3.Nd4 Nc6'],
      shape=(4780,), dtype=object)

In [24]:
games["date"].is_monotonic_increasing

True

In [25]:
games['opening_simple']=games['opening'].str.split().str[:2].str.join(' ').str.split('.').str[:1].str.join(' ')
games['opening_simple'].value_counts()

opening_simple
Sicilian Defense    1348
Kings Indian         622
English Opening      579
Queens Gambit        510
Queens Pawn          465
                    ... 
Ponziani Opening       4
Englund Gambit         4
English Defense        3
Richter Veresov        2
Kadas Opening          1
Name: count, Length: 70, dtype: int64

___
## 7. Defining Game Archetypes

In [26]:
g= games['magnus_accuracy']
print(g.sort_values().iat[int(g.shape[0]*0.75)],
g.sort_values().iat[int(g.shape[0]*0.25)]) #my way
print(g.quantile(0.75), g.quantile(0.25)) #fundamental way

93.84 85.54
93.84 85.5425


To evaluate whether Magnus Carlsen thrives in high-precision or chaotic environments, we classify games as **Clean** ($\ge 93.84\%$) or **Messy** ($\le 85.54\%$) using the 75th and 25th percentiles of accuracy. We apply these exact cutoffs universally to both players rather than using separate opponent thresholds because chess engine accuracy is an objective measure of move quality—an $88\%$ game reflects the same level of tactical chaos regardless of who plays. Using a single standard lets us fairly evaluate Magnus's performance across four distinct game environments (Clean, Messy, and Asymmetric).

In [27]:
games['opponent_accuracy'] = np.where(
    games['magnus_color'] == 'white', 
    games['black_accuracy'], 
    games['white_accuracy']
)

games['opponent_accuracy']

0       82.04
1       80.40
2       76.70
3       85.41
4       87.73
        ...  
9331    84.14
9332    81.43
9333    73.50
9334    80.18
9335    95.67
Name: opponent_accuracy, Length: 9326, dtype: float64

In [28]:
high_accuracy_threshold = 93.84
low_accuracy_threshold = 85.54

conditions = [
    (games['magnus_accuracy'] >= high_accuracy_threshold) & (games['opponent_accuracy'] >= high_accuracy_threshold),
    (games['magnus_accuracy'] <= low_accuracy_threshold) & (games['opponent_accuracy'] <= low_accuracy_threshold)
]

choices = ['clean', 'messy']

games['game_type'] = np.select(conditions, choices, default='mixed')

games['game_type'].value_counts()

game_type
mixed    7171
messy    1666
clean     489
Name: count, dtype: int64

___
## 8. Rating sanity check


In [29]:
# 1. Check for missing values
print("--- Missing Rating Counts ---")
print(games[['magnus_rating', 'opponent_rating']].isna().sum())

# 2. Check summary stats (look closely at min, max, and percentiles)
print("\n--- Rating Summary Statistics ---")
print(games[['magnus_rating', 'opponent_rating']].describe().round(1))


--- Missing Rating Counts ---
magnus_rating      0
opponent_rating    0
dtype: int64

--- Rating Summary Statistics ---
       magnus_rating  opponent_rating
count         9326.0           9326.0
mean          3235.1           2979.8
std            122.4            262.6
min           2692.0            259.0
25%           3213.0           2911.0
50%           3265.0           3037.0
75%           3309.0           3121.0
max           3401.0           3459.0


Magnus's rating range (2692–3401) is consistent with known history. However, 
opponent ratings range as low as 259 with high variance (σ = 262 vs Magnus's 
σ = 122), indicating a long tail of casual/unrated-style opponents mixed into 
the data — see filtering decision below.

Filtering to opponents rated ≥2112 (1st percentile) removes 103 likely casual/
exhibition games, reducing the dataset to 9,233 games for session and upset analysis.

In [30]:
games["opponent_rating"].quantile(0.01)  # check what the bottom 1% looks like

np.float64(2112.0)

In [31]:
games = games[games['opponent_rating'] >= 2112].copy()
games.shape

(9233, 28)

___
## 9. Session and fatigue setup
Games played within 30 minutes of each other are grouped into a "session" — 
a proxy for a single sitting. This enables checking whether accuracy declines 
as a session goes on (Q3).

In [32]:
# 1. Sort chronologically by your existing 'date' column and reset the index
games = games.sort_values('date').reset_index(drop=True)

# 2. Calculate time gap in minutes between consecutive games
games['time_since_last_game_min'] = games['date'].diff().dt.total_seconds() / 60.0

# 3. Flag new session (> 30 minute gap or first game of the dataset)
games['is_new_session'] = (games['time_since_last_game_min'] > 30) | (games['time_since_last_game_min'].isna())

# 4. Generate session_id and game sequence numbers
games['session_id'] = games['is_new_session'].cumsum() #the True(1) and False(0), so it keeps on summing
games['game_number_in_session'] = games.groupby('session_id').cumcount() + 1  #only works with groupby and just counts the rows not the content in it.

# Quick check
print("Session indexing complete!")
print(f"Total Games Analyzed:     {len(games)}")
print(f"Total Playing Sessions:    {games['session_id'].nunique()}")
print(f"Max Games in Single Binge: {games['game_number_in_session'].max()}")

Session indexing complete!
Total Games Analyzed:     9233
Total Playing Sessions:    917
Max Games in Single Binge: 115


____
## 10. Detecting upsets
An "upset" is defined as a loss where Magnus held a rating advantage in the 
top 10% of all his losses (>=280.4 points) — using his own loss distribution 
rather than an arbitrary fixed number, so the threshold is grounded in his 
actual results.

In [33]:
games['rating_diff']=games['magnus_rating']-games['opponent_rating']
games['rating_diff']

0       502
1       478
2       527
3       653
4       282
       ... 
9228    312
9229    341
9230    345
9231    349
9232    337
Name: rating_diff, Length: 9233, dtype: int64

In [34]:
# Filter only games Magnus lost
losses = games[games['magnus_result_simple'] == 'loss'].copy()

# Look at the distribution of the rating gap when he loses
q90 = games[games['magnus_result_simple'] == 'loss']['rating_diff'].quantile(0.90)
print(losses['rating_diff'].describe(percentiles=[0.5, 0.75, 0.90]), q90)

count    1617.000000
mean      141.427953
std       132.172095
min      -333.000000
50%       140.000000
75%       203.000000
90%       280.400000
max      1107.000000
Name: rating_diff, dtype: float64 280.4000000000001


In [35]:
games['is_upset']=(games['magnus_result_simple']=='loss') & (games['rating_diff']>=q90)
games['is_upset']

0        True
1       False
2       False
3       False
4       False
        ...  
9228    False
9229    False
9230    False
9231    False
9232    False
Name: is_upset, Length: 9233, dtype: bool

___
## 11. Filtering to standard chess

In [36]:
games.groupby('rules')['time_class'].value_counts()

rules     time_class
chess     blitz         6540
          bullet        2096
          rapid          315
chess960  blitz          275
          bullet           7
Name: count, dtype: int64


Chess960, odds chess, and three-check games use different rules and starting 
positions, making opening/rating analysis incompatible with standard chess. 
These are excluded from the main analysis (~316 games, ~3.4%).

Note: `session_id` and `game_number_in_session` were computed on the full 
dataset (including non-standard variants) to preserve real chronological 
continuity. After filtering to standard chess, some gaps in game numbering 
are expected — they indicate a non-standard game occurred at that point in 
the session, not a data error.

In [37]:
chess_games = games[games["rules"] == "chess"].copy()
chess_games

,url,time_class,time_control,rated,rules,end_time,eco_url,opening,white_username,white_rating,...,prev_result,opening_simple,opponent_accuracy,game_type,time_since_last_game_min,is_new_session,session_id,game_number_in_session,rating_diff,is_upset
0,https://www.chess.com/game/live/998801568,rapid,1500+25,False,chess,1418584335,https://www.chess.com/openings/Trompowsky-Atta...,Trompowsky Attack 2...c5 3.d5 Ne4,MagnusCarlsen,2862,...,win,Trompowsky Attack,87.73,mixed,NaN,True,1,1,502,True
1,https://www.chess.com/game/live/998801336,rapid,1500+25,False,chess,1418584494,https://www.chess.com/openings/Queens-Gambit-D...,Queens Gambit Declined Harrwitz Two Knights Bl...,MagnusCarlsen,2862,...,win,Queens Gambit,80.87,mixed,2.650000,False,1,2,478,False
2,https://www.chess.com/game/live/998801191,rapid,1500+25,False,chess,1418585285,https://www.chess.com/openings/Indian-Game-Paw...,Indian Game Pawn Indian Variation 3.Nc3 c5 4.d...,MagnusCarlsen,2862,...,win,Indian Game,85.14,mixed,13.183333,False,1,3,527,False
3,https://www.chess.com/game/live/998801933,rapid,1500+25,False,chess,1418585478,https://www.chess.com/openings/Birds-Opening-D...,Birds Opening Dutch Variation 2.Nf3,MagnusCarlsen,2862,...,win,Birds Opening,81.89,mixed,3.216667,False,1,4,653,False
5,https://www.chess.com/game/live/1627677596,blitz,300+2,True,chess,1466702438,https://www.chess.com/openings/Kings-Indian-At...,Kings Indian Attack 2...c6 3.Bg2 Bf5 4.d3,TigranLPetrosyan,2605,...,win,Kings Indian,90.71,mixed,9.100000,False,2,2,374,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9228,https://www.chess.com/game/live/169139813150,blitz,180,True,chess,1779569185,https://www.chess.com/openings/Nimzo-Indian-De...,Nimzo Indian Defense Kmoch Variation 4...c5 5.a3,MagnusCarlsen,3373,...,win,Nimzo Indian,84.14,mixed,3.083333,False,916,12,312,False
9229,https://www.chess.com/game/live/169410789930,blitz,180,True,chess,1780061104,https://www.chess.com/openings/Pirc-Defense-2....,Pirc Defense 2.d4 Nf6 3.Bd3 e5 4.Nf3,Resentiment18,3034,...,win,Pirc Defense,81.43,mixed,8198.650000,True,917,1,341,False
9230,https://www.chess.com/game/live/169410964060,blitz,180,True,chess,1780061380,https://www.chess.com/openings/English-Opening...,English Opening Kings English Nimzowitsch Floh...,MagnusCarlsen,3377,...,win,English Opening,73.50,messy,4.600000,False,917,2,345,False
9231,https://www.chess.com/game/live/169411164842,blitz,180,True,chess,1780061522,https://www.chess.com/openings/Philidor-Defens...,Philidor Defense Exchange Variation...6.Bf4 O ...,Resentiment18,3030,...,win,Philidor Defense,80.18,mixed,2.366667,False,917,3,349,False


___
## 12. Saving Cleaned Data

In [38]:
games.to_csv("magnus_cleaned_games.csv", index=False)

In [39]:
chess_games.to_csv("magnus_cleaned_chess_games.csv", index=False)